In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# --- 1. Flatten all NDVI and LST values across all years ---
ndvi = ndvi_stack.flatten()
lst = lst_stack.flatten()

# --- 2. Rescale NDVI if necessary ---
# NDVI values should be between -1 and +1
if np.nanmax(ndvi) > 1.5:
    ndvi = ndvi / 1000.0  # adjust scaling if NDVI was stored as integers

# --- 3. Convert LST from Kelvin to Celsius ---
lst_celsius = lst - 273.15

# --- 4. Mask invalid or unrealistic values ---
mask = (
    np.isfinite(ndvi) &
    np.isfinite(lst_celsius) &
    (ndvi > -0.2) & (ndvi < 0.9) &   # realistic NDVI range
    (lst_celsius > 10) & (lst_celsius < 60)  # realistic LST range in °C
)

ndvi_clean = ndvi[mask]
lst_clean = lst_celsius[mask]

print("Valid pixels:", len(ndvi_clean))

# --- 5. Subsample for plotting ---
np.random.seed(42)
idx = np.random.choice(len(ndvi_clean), size=min(20000, len(ndvi_clean)), replace=False)
x = ndvi_clean[idx]
y = lst_clean[idx]

# --- 6. Regression line ---
slope, intercept = np.polyfit(x, y, 1)
x_line = np.linspace(-0.2, 0.9, 100)
y_line = slope * x_line + intercept

# --- 7. Correlation ---
corr = np.corrcoef(x, y)[0, 1]
r2 = corr**2

print(f"Correlation (r): {corr:.3f}")
print(f"R²: {r2:.3f}")

# --- 8. Plot ---
plt.figure(figsize=(8,6))
plt.scatter(x, y, s=2, alpha=0.15, label="Pixel samples")
plt.plot(x_line, y_line, color="red", linewidth=3, label="Linear fit")

plt.xlabel("NDVI")
plt.ylabel("LST (°C)")
plt.title("Overall NDVI–LST Relationship (1985–2024)")

plt.text(
    0.02, 0.95,
    f"r = {corr:.2f}\nR² = {r2:.2f}",
    transform=plt.gca().transAxes,
    fontsize=12,
    bbox=dict(facecolor="white", alpha=0.7, edgecolor="none")
)

plt.grid(True)
plt.legend()
plt.tight_layout()
plt.show()
